# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**NICOLAS LASTRA**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi, ActionHanoi
from aima_libs.tree_hanoi import NodeHanoi
from aima_libs.aima import PriorityQueue
import numpy as np

In [2]:
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA
    

    'INICIO DEFINICION DE FUNCIONES A UTILIZAR EN LA BUSQUEDA'
    
    #Creamos una funcion la cual cuantifica cual es la diferencia entre un nodo de referencia/actual y otro de interes
    def numero_de_diferencias_entre_nodos(nodo_referencia: NodeHanoi, nodo_a_comparar: NodeHanoi) -> int:
    
        cantidad_diferencias = 0
        
        #Verifica que la longitud de las varillas sean iguales
        for i in range(3):
        
            diferencia_longitud =  len(nodo_referencia.state.rods[i]) - len(nodo_a_comparar.state.rods[i])
            
    
            # la varilla del nodo de referencia tiene mas elementos que el nodo a comparar
            if(diferencia_longitud > 0): 
                for k in range(len(nodo_referencia.state.rods[i])):
                    if (nodo_referencia.state.rods[i][k] not in nodo_a_comparar.state.rods[i]):
                        cantidad_diferencias += 1
                    
            # la varilla del nodo de referencia tiene menos elementos que el nodo a comparar           
            if(diferencia_longitud < 0): 
                for k in range(len(nodo_a_comparar.state.rods[i])):
                    if (nodo_a_comparar.state.rods[i][k] not in nodo_referencia.state.rods[i]):
                        cantidad_diferencias += 1
    
            # la varilla del nodo de referencia tiene la misma cantidad de elementos que el nodo a comparar
            if(diferencia_longitud == 0): 
                for k in range(len(nodo_referencia.state.rods[i])):
                    if (nodo_referencia.state.rods[i][k] not in nodo_a_comparar.state.rods[i]):
                        cantidad_diferencias += 1
            
        return cantidad_diferencias

    #Creamos una funcion la cual nos arroja un nodo con su recorrido ideal, esta resuelto de forma recursiva, formando el camino de menor costo
    def solucion_ideal_torre_hanoi(numero_de_discos: int) -> NodeHanoi:
        
        #Inicializamos variables utilizadas segun la cantidad de discos.
        numero_de_discos_totales = numero_de_discos
        
        #Se define la cantidad teorica o ideal de movimientos para resolver el problema.
        cantidad_movimientos_teoricos = (2 ** numero_de_discos_totales)-1
    
        #Se define una lista de arreglos de NumPy para los movimientos de cada disco con su frecuencia y direccion.
        movimientos_discos = [None] * (numero_de_discos + 1)
    
        #Se define un arreglo distinto para la direccion de cada disco (izquierda / derecha)
        direcciones = np.empty(numero_de_discos + 1, dtype=int)
    
        #Genera la secuencia con el orden de movimiento para cada disco.
        for numero_de_disco in range(1,numero_de_discos_totales+1,1):
            paso_tiempo = 2**numero_de_disco #En que numero de movimiento se debe mover este disco.
            primer_movimiento = 2**(numero_de_disco-1) #Cual es el numero de movimiento en el que se mueve por primera vez. 
            movimientos_discos[numero_de_disco] = np.arange(primer_movimiento, cantidad_movimientos_teoricos + 1, paso_tiempo)
    
        #Se define la direccion de los movimientos para cada disco (0: Sin direccion asociada / -1: izquierda / 1: derecha)
        indices = np.arange(numero_de_discos_totales + 1)
        direcciones = np.where(((indices % 2 == 0) & (numero_de_discos_totales % 2 == 0)) | ((indices % 2 != 0) & (numero_de_discos_totales % 2 != 0)), -1, 1)
        direcciones[0] = 0
        
        #Define la secuencia de movimientos
        secuencia = np.arange(1,cantidad_movimientos_teoricos+1,1)
    
        #Inicializa una matriz donde(indice: numero de movimiento / columna 0: Numero de disco a mover / columna 1: Direccion del movimiento 
        matriz_secuencia_movimientos = np.empty((cantidad_movimientos_teoricos, 2))
            
        for i, valor in enumerate(secuencia):
            for disco, lista_movimientos_disco in enumerate(movimientos_discos):
                if (np.isin(valor, lista_movimientos_disco)):
                    matriz_secuencia_movimientos[i][0]=disco
                    matriz_secuencia_movimientos[i][1]=direcciones[disco]
                    
        #(indice: numero de movimiento / columna 0: Numero de disco a mover / columna 1: Direccion del movimiento 
        matriz_secuencia_movimientos = matriz_secuencia_movimientos.astype(int) 
    
        #Se inicializa 
        lista_discos = [i for i in range(numero_de_discos, 0, -1)]
        estado_inicial = StatesHanoi(lista_discos,[],[],max_disks=numero_de_discos)
        estado_objetivo = StatesHanoi([],[],lista_discos,max_disks=numero_de_discos)
        problema = ProblemHanoi(estado_inicial,estado_objetivo)
        nodo_actual = NodeHanoi(problema.initial)
        
        #Determino que disco voy a mover y su direccion.
        for disco_a_mover, disco_a_mover_direccion in matriz_secuencia_movimientos:
            disco_a_mover.item()
            disco_a_mover_direccion.item()
            
            #Determino en donde esta el disco que quiero mover en el nodo actual y lo muevo segun la secuencia.
            for numero_varilla, info_varilla in enumerate(nodo_actual.state.rods):
        
                if (disco_a_mover in info_varilla):
                    varilla_origen = numero_varilla
                    varilla_destino = varilla_origen + disco_a_mover_direccion 
                    
                    if(varilla_destino==3):
                        varilla_destino=0
                    if(varilla_destino==-1):
                        varilla_destino=2
        
                    estado_siguiente = ActionHanoi(disk=disco_a_mover, rod_input=varilla_origen,rod_out=varilla_destino).execute(nodo_actual.state)
                    nodo_siguiente = nodo_actual.expand(problema)
                    
                    for i, nodo_i in enumerate(nodo_siguiente):
                        if (nodo_i.state == estado_siguiente):
                            nodo_actual = nodo_i
                            break
                    if (nodo_actual.state == estado_siguiente):
                        break
                        
        nodo_ideal = nodo_actual
        
        return nodo_ideal

    #Creamos una funcion heuristica que evalua a traves de dos parametros (diferencia con respecto al camino ideal y numero de paso dentro del 
    #camino ideal, ya que a igualdad de diferencias es mejor la opcion mas proxima a la solucion.
    def evaluar_nodo(nodo_frontera: list) -> (int, list):
    
        #Lista que contiene la diferencia del nodo a evaluar con cada nodo que forma parte del camino ideal a la resolucion
        lista_diferencias_camino_ideal=[]
        
         #Creamos el camino ideal, con la menor cantidad de pasos/costo para tomar como referencia
        #nodo_ideal = solucion_ideal_torre_hanoi(number_disks)
        path_ideal = nodo_ideal.path()
        path_ideal.pop(0)
        
        for nodo_i in (path_ideal):
            lista_diferencias_camino_ideal.append(numero_de_diferencias_entre_nodos(nodo_frontera,nodo_i))
    
        valor_evaluado = lista_diferencias_camino_ideal.index(min(lista_diferencias_camino_ideal)) - min(lista_diferencias_camino_ideal)
    
        return valor_evaluado

    
    'FIN DEFINICION DE FUNCIONES A UTILIZAR EN LA BUSQUEDA'
    #Definimos el mejor camino a seguir en funcion de la cantidad de discos, el cual va servir como referencia de que tan lejos esta cada nodo
    #dentro de la funcion heuristica
    nodo_ideal = solucion_ideal_torre_hanoi(number_disks)
    
    
    #Creamos una cola prioritaria donde el valor de la funcion va ser la funcion heuristica
    frontera = PriorityQueue(order='max', f=evaluar_nodo)

    
    # Inicializamos las salidas, pero reemplazar con lo que se quiera usar.
    metrics = {
        "solution_found": True,
        "nodes_explored": None,
        "states_visited": None,
        "nodes_in_frontier": None,
        "max_depth": None,
        "cost_total": None,
    }

    #Se crea el nodo raiz y se coloca en la frontera
    frontera.append(NodeHanoi(initial_state))
    solucion_encontrada = False
    numero_nodos_explorados = 0
    nodos_explorados = set()
    
    while len(frontera) != 0:
        solution = frontera.pop()[1] #Se quita el nodo con mayor prioridad.
        numero_nodos_explorados += 1
        nodos_explorados.add(solution.state)
        
        if problem.goal_test(solution.state):
            solucion_encontrada = True
            print("Encontramos la solución")
            break
                
        # Agregamos a la frontera los nodos sucesores que no hayan sido visitados
        for next_node in solution.expand(problem):
            if next_node.state not in nodos_explorados:
                frontera.append(next_node)
            
    if not solucion_encontrada:
        print("No se encontró la solución")
        
    # TODO: Completar con el algoritmo de búsqueda que desees implementar
    #####

    metrics['solution_found'] = solucion_encontrada
    metrics["nodes_explored"] = numero_nodos_explorados
    metrics["states_visited"] = len(nodos_explorados)
    metrics["nodes_in_frontier"] = len(frontera)
    metrics["max_depth"] = solution.depth
    metrics["cost_total"] = solution.state.accumulated_cost
    
    return solution, metrics

Se prueba la función:

In [3]:
solution, metrics = search_algorithm(number_disks=5)

Encontramos la solución


Veamos las métricas:

In [4]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 32
states_visited: 32
nodes_in_frontier: 31
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [5]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [6]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3


### La funcion heurística en este caso es f->"evaluar_nodo":
Esta funcion evalua el nodo pasado como argumento y calcula la diferencia minima del nodo actual con el nodo mas cercano a lo largo del camino ideal de la resolucion del problema de hanoi (se encuentra el camino ideal de forma recursiva). La funcion devuelve un valor que es compuesto por la diferencia minima descripta y el indice de mayor cercania con el nodo final de la solucion, de esta forma evitamos redundancias en caso de que dos nodos del camino tengan la misma diferencia con el nodo evaluado.

En este caso tambien vamos a ver que el camino coincide con el ideal porque partimos de la misma condicion inicial y final que se considero para calcular la recursividad, pero si le cambiamos las condiciones de inicio veremos otros resultados, ya que las metricas no van a ser las ideales, lo cual pone en evidencia que se toma el camino de recursividad solo como herramienta para la heuristica utilizada para llegar a la solucion.